In [ ]:
!pip install albumentations opencv-python-headless numpy Pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify karo
import os
print(os.listdir('/content/drive/MyDrive/CargoDeep/Animal_smuggling'))
# Output hona chahiye: ['reptile', 'bird', 'mammal', 'organic_mass']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['reptile', 'organic-mass', 'mammals', 'bird']


In [ ]:
# ✅ BUS YAHAN PATH CHANGE KARO — BAAKI KUCH MAT CHHUO
INPUT_DIR  = "/content/drive/MyDrive/CargoDeep/Animal_smuggling"
OUTPUT_DIR = "/content/drive/MyDrive/CargoDeep/augmented_dataset"

# Verify
for cls in ['reptile', 'bird', 'mammals', 'organic-mass']:
    path = f"{INPUT_DIR}/{cls}"
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"  {cls}: {count} images")

  reptile: 12 images
  bird: 5 images
  mammals: 5 images
  organic-mass: 6 images


In [ ]:
import os, cv2, shutil, random
import numpy as np
from tqdm import tqdm
import albumentations as A


AUGMENT_PER_IMAGE = 100   # 5 images × 100 = 500 per class
TRAIN_RATIO       = 0.8
IMG_SIZE          = 640

CLASSES = {
    "reptile":      0,
    "bird":         1,
    "mammal":       2,
    "organic_mass": 3,
}

IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# ── AUGMENTATION PIPELINE ──────────────────────────
def get_augmentation_pipeline():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Rotate(limit=20, p=0.6),
        A.Affine(
            scale=(0.8, 1.2),
            translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
            shear=(-10, 10),
            p=0.5
        ),
        A.RandomResizedCrop(
            size=(IMG_SIZE, IMG_SIZE),
            scale=(0.7, 1.0),
            p=0.4
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.3,
            contrast_limit=0.3,
            p=0.7
        ),
        A.GaussNoise(p=0.4),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.CLAHE(clip_limit=3.0, p=0.4),
        A.RandomGamma(gamma_limit=(70, 130), p=0.4),
        A.Sharpen(alpha=(0.1, 0.4), lightness=(0.8, 1.2), p=0.3),
    ])


def auto_detect_bbox(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return [0.5, 0.5, 0.8, 0.8]
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    masks = [
        cv2.inRange(hsv, np.array([0,  40, 40]), np.array([15, 255, 255])),
        cv2.inRange(hsv, np.array([15, 40, 40]), np.array([35, 255, 255])),
    ]
    mask = cv2.bitwise_or(masks[0], masks[1])
    kernel = np.ones((20, 20), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return [0.5, 0.5, 0.8, 0.8]
    valid = [c for c in contours if cv2.contourArea(c) > (w * h * 0.005)]
    if not valid:
        return [0.5, 0.5, 0.8, 0.8]
    all_pts = np.vstack(valid)
    x, y, bw, bh = cv2.boundingRect(all_pts)
    return [
        round((x + bw/2) / w, 6),
        round((y + bh/2) / h, 6),
        round(bw / w, 6),
        round(bh / h, 6),
    ]

def save_label(label_path, class_id, bbox):
    x, y, w, h = [max(0.001, min(0.999, v)) for v in bbox]
    with open(label_path, 'w') as f:
        f.write(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

def save_sample(img_arr, bbox, class_id, fname, split):
    img_bgr = cv2.cvtColor(np.array(img_arr), cv2.COLOR_RGB2BGR)
    cv2.imwrite(
        f"{OUTPUT_DIR}/images/{split}/{fname}.jpg",
        img_bgr,
        [cv2.IMWRITE_JPEG_QUALITY, 95]
    )
    save_label(
        f"{OUTPUT_DIR}/labels/{split}/{fname}.txt",
        class_id, bbox
    )


def run():
    # Clean + create output dirs
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    for split in ["train", "val"]:
        os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)

    transform = get_augmentation_pipeline()
    print(f"✅ Pipeline ready — {AUGMENT_PER_IMAGE}x augmentation per image\n")

    for class_name, class_id in CLASSES.items():
        class_dir = f"{INPUT_DIR}/{class_name}"
        if not os.path.exists(class_dir):
            print(f"⚠️  {class_name} folder nahi mila — skip")
            continue

        images = [f for f in os.listdir(class_dir)
                  if f.lower().endswith(IMG_EXTENSIONS)]

        if not images:
            print(f"⚠️  {class_name} mein images nahi — skip")
            continue

        print(f"📂 {class_name} [{class_id}] — {len(images)} images → {len(images)*AUGMENT_PER_IMAGE} expected")
        all_samples = []

        for img_file in tqdm(images, desc=f"  {class_name}"):
            img_path = f"{class_dir}/{img_file}"
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            bbox = auto_detect_bbox(img_path)

            base = os.path.splitext(img_file)[0]
            all_samples.append((img_rgb, bbox, f"orig_{class_name}_{base}"))

            for i in range(AUGMENT_PER_IMAGE):
                try:
                    aug = transform(image=img_rgb)['image']
                    all_samples.append((aug, bbox, f"aug{i:04d}_{class_name}_{base}"))
                except:
                    continue

        random.shuffle(all_samples)
        split_idx = int(len(all_samples) * TRAIN_RATIO)
        train_set = all_samples[:split_idx]
        val_set   = all_samples[split_idx:]

        for img_arr, bbox, fname in train_set:
            save_sample(img_arr, bbox, class_id, fname, "train")
        for img_arr, bbox, fname in val_set:
            save_sample(img_arr, bbox, class_id, fname, "val")

        print(f"   ✅ Train: {len(train_set)} | Val: {len(val_set)}\n")

    # Save dataset.yaml
    yaml = f"""path: {OUTPUT_DIR}
train: images/train
val:   images/val
nc: {len(CLASSES)}
names:
"""
    for name, idx in sorted(CLASSES.items(), key=lambda x: x[1]):
        yaml += f"  {idx}: {name}\n"

    with open(f"{OUTPUT_DIR}/dataset.yaml", 'w') as f:
        f.write(yaml)

    # Final stats
    train_count = len(os.listdir(f"{OUTPUT_DIR}/images/train"))
    val_count   = len(os.listdir(f"{OUTPUT_DIR}/images/val"))
    print("═"*45)
    print(f"  TRAIN : {train_count} images")
    print(f"  VAL   : {val_count}   images")
    print(f"  TOTAL : {train_count + val_count} images")
    print("═"*45)
    print(f"✅ dataset.yaml saved at: {OUTPUT_DIR}/dataset.yaml")

run()

✅ Pipeline ready — 100x augmentation per image

📂 reptile [0] — 11 images → 1100 expected


  reptile: 100%|██████████| 11/11 [00:16<00:00,  1.52s/it]


   ✅ Train: 888 | Val: 223

📂 bird [1] — 5 images → 500 expected


  bird: 100%|██████████| 5/5 [00:08<00:00,  1.70s/it]


   ✅ Train: 404 | Val: 101

⚠️  mammal folder nahi mila — skip
⚠️  organic_mass folder nahi mila — skip
═════════════════════════════════════════════
  TRAIN : 1292 images
  VAL   : 324   images
  TOTAL : 1616 images
═════════════════════════════════════════════
✅ dataset.yaml saved at: /content/drive/MyDrive/CargoDeep/augmented_dataset/dataset.yaml


In [ ]:

import os

output = "/content/drive/MyDrive/CargoDeep/augmented_dataset"

for split in ["train", "val"]:
    imgs   = len(os.listdir(f"{output}/images/{split}"))
    labels = len(os.listdir(f"{output}/labels/{split}"))
    print(f"{split}: {imgs} images, {labels} labels")

# YAML print karo
with open(f"{output}/dataset.yaml") as f:
    print(f.read())

train: 1292 images, 1292 labels
val: 324 images, 324 labels
path: /content/drive/MyDrive/CargoDeep/augmented_dataset
train: images/train
val:   images/val
nc: 4
names:
  0: reptile
  1: bird
  2: mammal
  3: organic_mass



In [ ]:

!pip install ultralytics -q

from ultralytics import YOLO
import torch


print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


DATA_YAML = "/content/drive/MyDrive/CargoDeep/augmented_dataset/dataset.yaml"

model = YOLO('yolov8m.pt')  # Medium model — good balance


results = model.train(
    data      = DATA_YAML,
    epochs    = 40,
    imgsz     = 640,
    batch     = 8,
    patience  = 20,
    save      = True,
    plots     = True,
    name      = 'xray_smuggling_v1',
    device    = 0,
    workers   = 2,
    lr0       = 0.01,
    lrf       = 0.001,
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    cos_lr    = True,
    verbose   = True,
)

print("✅ Training complete!")
print(f"Best model saved at: {results.save_dir}")

GPU available: True
Device: Tesla T4
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/CargoDeep/augmented_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=xray_smuggling_v12, nbs=64, nms=False, opset=No

In [2]:
# Cell 1 — Install only
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.8 MB/s eta 0:00:00


In [3]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
MODEL_PATH = '/content/runs/detect/xray_smuggling_v12/weights/best.pt'
print(f"Updated MODEL_PATH to: {MODEL_PATH}")

Updated MODEL_PATH to: /content/runs/detect/xray_smuggling_v12/weights/best.pt


In [10]:
script_path = '/content/drive/MyDrive/CargoDeep/xray_predict.py'
with open(script_path, 'r') as f:
    script_content = f.read()

# Replace the hardcoded MODEL_PATH in the script content
old_model_path = 'MODEL_PATH = "/content/drive/MyDrive/CargoDeep/model/best.pt"'
new_model_path = "MODEL_PATH = '/content/runs/detect/xray_smuggling_v12/weights/best.pt'"
modified_script_content = script_content.replace(old_model_path, new_model_path)

# Execute the modified script content
exec(modified_script_content)

  X-Ray Animal Smuggling — Prediction & Evaluation

  ERROR: Model not found at:
  /content/runs/detect/xray_smuggling_v12/weights/best.pt

  Check your Drive path and try again.


In [11]:
import os
model_actual_path = '/content/runs/detect/xray_smuggling_v12/weights/best.pt'

if os.path.exists(model_actual_path):
    print(f"✅ Model found at: {model_actual_path}")
else:
    print(f"❌ Model NOT found at: {model_actual_path}")
    print("Please check the path and ensure the training completed successfully.")

❌ Model NOT found at: /content/runs/detect/xray_smuggling_v12/weights/best.pt
Please check the path and ensure the training completed successfully.


In [12]:
import os
output_weights_dir = '/content/runs/detect/xray_smuggling_v12/weights/'

print(f"Listing contents of: {output_weights_dir}")
if os.path.exists(output_weights_dir):
    for item in os.listdir(output_weights_dir):
        print(item)
else:
    print(f"❌ Directory NOT found: {output_weights_dir}")

Listing contents of: /content/runs/detect/xray_smuggling_v12/weights/
❌ Directory NOT found: /content/runs/detect/xray_smuggling_v12/weights/


In [9]:
script_path = '/content/drive/MyDrive/CargoDeep/xray_predict.py'
with open(script_path, 'r') as f:
    script_content = f.read()
print(script_content)

"""
X-Ray Animal Smuggling Detection
Prediction & Evaluation Script

- Loads already trained model (best.pt)
- No retraining needed
- Tests on sample images
- Plots confusion matrix

Classes:
    0 - reptile      (lizard, snake, turtle)
    1 - bird         (parrot, exotic birds)

Requirements:
    pip install ultralytics
"""

import os
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from google.colab import files


# =============================================================
#  CONFIGURATION — Only change these paths
# =============================================================

MODEL_PATH = "/content/drive/MyDrive/CargoDeep/model/best.pt"
DATA_YAML  = "/content/drive/MyDrive/CargoDeep/augmented_dataset/dataset.yaml"
TEST_FOLDER= "/content/drive/MyDrive/CargoDeep/dataset"  # source images for testing

CLASS_NAMES = {
    0: "reptile",
    1: "bird",
}

CONFIDENCE = 0.25  # Detection threshold


# ================================================